# Brain tumor MRI: model comparison on a free Colab GPU

**Before running:**
1. `Runtime -> Change runtime type -> T4 GPU -> Save`.
2. Upload `processed.zip` (made on your laptop, see `docs/COLAB.md`) to Google Drive at `My Drive/btmri/processed.zip`.

Then run the cells from top to bottom (`Runtime -> Run all`). All results are saved to `My Drive/btmri/runs`, so nothing is lost if Colab disconnects.

In [ ]:
# 1. Check that a GPU is attached (you should see a table mentioning "Tesla T4").
!nvidia-smi

In [ ]:
# 2. Connect Google Drive (a pop-up asks for permission).
from google.colab import drive

drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/btmri"
RUNS = f"{DRIVE}/runs"

In [ ]:
# 3. Get the code (or update it if this session already has it) and install it.
import os

REPO = "https://github.com/sallew-glitch/brain-tumor-mri-gradcam.git"
if not os.path.exists("/content/project"):
    !git clone -q {REPO} /content/project
%cd /content/project
!git pull -q
!pip install -q -e ".[dev]"

In [ ]:
# 4. Unpack the data onto Colab's fast local disk, then run the tests.
!mkdir -p data && unzip -q -o {DRIVE}/processed.zip -d data/
!echo "images: $(ls data/processed/images | wc -l)  (expected 3064)"
!python -m pytest -q

In [ ]:
# 5. Train every model on fold 1 (roughly 20-40 minutes in total on a T4).
#    resnet18_cpu is the laptop baseline config; it runs fine on a GPU too.
MODELS = ["resnet18_cpu", "efficientnet_b0", "densenet121", "resnet50"]

for model in MODELS:
    !python scripts/train.py --config configs/{model}.yaml --folds 1 --set output_dir={RUNS}

In [ ]:
# 6. Grad-CAM on the VALIDATION fold at both layers, for every run.
import glob

for run in sorted(glob.glob(f"{RUNS}/*")):
    !python scripts/explain.py --run-dir "{run}" --split val --layers last penultimate

In [ ]:
# 7. Compare. Choose using the VAL_macro_f1 column and the Grad-CAM table (val split).
!python scripts/compare_runs.py --runs-dir {RUNS}

## After choosing a model

Run the full 5-fold cross-validation for the chosen model (edit the name), then the final Grad-CAM report on the test folds at the layer you picked:

```python
!python scripts/train.py --config configs/efficientnet_b0.yaml --set output_dir={RUNS}
!python scripts/explain.py --run-dir "{RUNS}/<new_run_name>" --split test --layers penultimate
```

To look at a heatmap image here:

```python
from IPython.display import Image
Image(f"{RUNS}/<run_name>/fold_1/gradcam_val_last/examples.png")
```